In [111]:
import sys, subprocess

try:
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    import pickle
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    import pickle


In [112]:
from keras.models import load_model

## load the trained model, scaler, and label encoder from the pickle file
model = load_model('ann_model.h5')

## Load the encoder and scaler
with open('onehot_encoder_geo.pkl', 'rb') as f:
    encoder = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

In [120]:
## Example input data

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure' : 3,
    'Balance' : 60000,
    'NumOfProducts' : 2,
    'HasCrCard' : 1,
    'IsActiveMember' : 1,
    'EstimatedSalary' : 50000
}
input_data

{'CreditScore': 600,
 'Geography': 'France',
 'Gender': 'Male',
 'Age': 40,
 'Tenure': 3,
 'Balance': 60000,
 'NumOfProducts': 2,
 'HasCrCard': 1,
 'IsActiveMember': 1,
 'EstimatedSalary': 50000}

In [121]:
#One-hot encode the 'Geography' column
geography_encoded = encoder.transform([[input_data['Geography']]]).toarray()
geography_encoded_df = pd.DataFrame(geography_encoded, columns=encoder.get_feature_names_out(['Geography']))
geography_encoded_df

e:\Users\rafay\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [122]:
input_df = pd.DataFrame([input_data])

In [123]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [124]:
## Encode Categorical Features
input_df['Gender'] = label_encoder.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [125]:
## Concatenate the one-hot and categorical encoded features with the rest of the input data
input_df = pd.concat([input_df.drop('Geography',axis=1), geography_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [126]:
## Scaling the input data
input_df_scaled = scaler.transform(input_df)

In [127]:
input_df_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [128]:
## Predict the output using the trained model
prediction = model.predict(input_df_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step


array([[0.04087085]], dtype=float32)

In [130]:
prediction_proba = prediction[0][0]

In [131]:
prediction_proba

np.float32(0.040870845)

In [132]:
if prediction_proba > 0.5:
    print(f"The customer is likely to leave the bank with a probability of {prediction_proba:.2f}.")
else:
    print(f"The customer is likely to stay with the bank with a probability of {1 - prediction_proba:.2f}.")

The customer is likely to stay with the bank with a probability of 0.96.
